1. SETUP, IMPORTAÇÕES E EXTRAÇÃO (DataSUS)

In [2]:
!pip install dbfread pyreaddbc pandas plotly requests pyarrow

import urllib.request
import os
import requests
import pandas as pd
import numpy as np
import plotly.express as px
from pyreaddbc import dbc2dbf
from dbfread import DBF

url = "ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOSP2021.dbc"
arquivo_dbc = "DOSP2021.dbc"
arquivo_dbf = "DOSP2021.dbf"

print("Baixando arquivo bruto do DataSUS...")
urllib.request.urlretrieve(url, arquivo_dbc)

print("Descompactando DBC para DBF...")
dbc2dbf(arquivo_dbc, arquivo_dbf)

print("Carregando no Pandas...")
tabela = DBF(arquivo_dbf, encoding='iso-8859-1')
df_sim = pd.DataFrame(iter(tabela))

print(f"✓ Carregado com sucesso: {df_sim.shape[0]:,} registros!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
Baixando arquivo bruto do DataSUS...
Descompactando DBC para DBF...
Carregando no Pandas...
✓ Carregado com sucesso: 431,616 registros!


In [ ]:
2. TRANSFORMAÇÃO E LIMPEZA

In [3]:
cols = ['DTOBITO', 'CAUSABAS', 'SEXO', 'IDADE', 'CODMUNRES', 'RACACOR', 'ESTCIV']
df = df_sim[cols].copy()

df['DTOBITO'] = pd.to_datetime(df['DTOBITO'], format='%d%m%Y', errors='coerce')

def calcular_idade_anos(cod):
    try:
        cod = str(int(float(cod))).zfill(3)
        unidade = cod[0]
        valor = int(cod[1:])
        if unidade == '4':
            return valor
        elif unidade in ['1', '2', '3']:
            return 0
        elif unidade == '5':
            return 100 + valor
        return np.nan
    except:
        return np.nan

df['IDADE_ANOS'] = df['IDADE'].apply(calcular_idade_anos)

df['SEXO_DESC'] = df['SEXO'].map({'1': 'Masculino', '2': 'Feminino', '0': 'Ignorado'}).fillna('Ignorado')
df['RACA_DESC'] = df['RACACOR'].map({
    '1': 'Branca', '2': 'Preta', '3': 'Amarela',
    '4': 'Parda', '5': 'Indígena'
}).fillna('Não Informado')

df['COD_MUN_6'] = df['CODMUNRES'].astype(str).str.slice(0, 6)

3. ENRIQUECIMENTO (API IBGE)

In [4]:
print("Buscando dados dos municípios de SP na API do IBGE...")
url_ibge = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/SP/municipios"
resposta = requests.get(url_ibge).json()

dim_municipios = pd.DataFrame([
    {
        'COD_MUN_6': str(m['id'])[:6],
        'NOME_MUNICIPIO': m['nome'],
        'MICRORREGIAO': m['microrregiao']['nome']
    } for m in resposta
])

df_analise = df.merge(dim_municipios, on='COD_MUN_6', how='left')
df_analise['NOME_MUNICIPIO'] = df_analise['NOME_MUNICIPIO'].fillna('Outro / Não Identificado')
print(f"✓ Base consolidada com {len(df_analise):,} registros!")

Buscando dados dos municípios de SP na API do IBGE...
✓ Base consolidada com 431,616 registros!


4. CLASSIFICAÇÃO EPIDEMIOLÓGICA (CID-10)

In [5]:
def refinar_grupo_cid(cid):
    if not isinstance(cid, str) or len(cid) == 0:
        return 'Não Identificado'

    cid = cid.strip().upper()
    letra = cid[0]

    if cid.startswith(('B342', 'U071', 'U072', 'B34.2', 'U07.1', 'U07.2')):
        return 'COVID-19'
    elif letra in ['A', 'B']:
        return 'Doenças Infecciosas e Parasitárias'
    elif letra in ['C', 'D0', 'D1', 'D2', 'D3', 'D4']:
        return 'Neoplasias (Câncer)'
    elif letra == 'E':
        return 'Doenças Endócrinas / Metabólicas (ex: Diabetes)'
    elif letra == 'I':
        return 'Doenças do Aparelho Circulatório'
    elif letra == 'J':
        return 'Doenças do Aparelho Respiratório'
    elif letra == 'K':
        return 'Doenças do Aparelho Digestivo'
    elif letra in ['V', 'W', 'X', 'Y']:
        return 'Causas Externas (Acidentes/Violência)'
    else:
        return 'Demais Causas'

df_analise['GRUPO_CID_REFINADO'] = df_analise['CAUSABAS'].apply(refinar_grupo_cid)

In [ ]:
5. INDICADORES DE SAÚDE (Mortalidade Prematura)

In [6]:
cond_idade_dcnt = (df_analise['IDADE_ANOS'] >= 30) & (df_analise['IDADE_ANOS'] <= 69)
dcnt_grupos = [
    'Doenças do Aparelho Circulatório',
    'Neoplasias (Câncer)',
    'Doenças do Aparelho Respiratório',
    'Doenças Endócrinas / Metabólicas (ex: Diabetes)'
]
cond_dcnt = df_analise['GRUPO_CID_REFINADO'].isin(dcnt_grupos)

df_analise['OBITO_PREMATURO_DCNT'] = np.where(cond_idade_dcnt & cond_dcnt, 1, 0)
df_analise['OBITO_PREMATURO_EXTERNA'] = np.where(
    (df_analise['IDADE_ANOS'] >= 15) & (df_analise['IDADE_ANOS'] <= 49) &
    (df_analise['GRUPO_CID_REFINADO'] == 'Causas Externas (Acidentes/Violência)'), 1, 0
)

total_prematuro = df_analise['OBITO_PREMATURO_DCNT'].sum()
pct_prematuro = (total_prematuro / len(df_analise)) * 100
print(f"• Óbitos Prematuros por DCNT (30 a 69 anos): {total_prematuro:,} ({pct_prematuro:.2f}% do total)")

• Óbitos Prematuros por DCNT (30 a 69 anos): 84,708 (19.63% do total)


6. RESUMO EXECUTIVO E TABELAS

In [7]:
total_obitos = len(df_analise)
idade_media = df_analise['IDADE_ANOS'].mean()
idade_mediana = df_analise['IDADE_ANOS'].median()

print("=" * 45)
print(" RESUMO EXECUTIVO - MORTALIDADE SP")
print("=" * 45)
print(f"• Total de Óbitos: {total_obitos:,}")
print(f"• Idade Média: {idade_media:.1f} anos")
print(f"• Idade Mediana: {idade_mediana:.1f} anos")
print("=" * 45)

ranking_causas = df_analise['GRUPO_CID_REFINADO'].value_counts().reset_index()
ranking_causas.columns = ['Grupo de Causa', 'Total de Óbitos']
ranking_causas['% do Total'] = (ranking_causas['Total de Óbitos'] / total_obitos) * 100
display(ranking_causas)

tabela_prematura = pd.crosstab(
    df_analise['SEXO_DESC'],
    df_analise['OBITO_PREMATURO_DCNT'],
    margins=True,
    normalize='index'
) * 100
tabela_prematura.columns = ['Demais Faixas/Causas (%)', 'Mortalidade Prematura DCNT (%)']
display(tabela_prematura.round(2))

 RESUMO EXECUTIVO - MORTALIDADE SP
• Total de Óbitos: 431,616
• Idade Média: 67.0 anos
• Idade Mediana: 69.0 anos


,Grupo de Causa,Total de Óbitos,% do Total
0,COVID-19,110401,25.578524
1,Doenças do Aparelho Circulatório,93115,21.573575
2,Demais Causas,61453,14.237887
3,Neoplasias (Câncer),57128,13.235839
4,Doenças do Aparelho Respiratório,36384,8.429715
5,Causas Externas (Acidentes/Violência),23103,5.352675
6,Doenças Endócrinas / Metabólicas (ex: Diabetes),19522,4.523002
7,Doenças do Aparelho Digestivo,18042,4.180105
8,Doenças Infecciosas e Parasitárias,12468,2.888679


,Demais Faixas/Causas (%),Mortalidade Prematura DCNT (%)
SEXO_DESC,,
Feminino,81.83,18.17
Ignorado,96.33,3.67
Masculino,79.15,20.85
All,80.37,19.63


7. VISUALIZAÇÕES GRÁFICAS (Plotly)

In [8]:
fig_causas = px.bar(
    ranking_causas.sort_values(by='Total de Óbitos', ascending=True),
    x='Total de Óbitos', y='Grupo de Causa', orientation='h',
    text='% do Total', title='<b>Distribuição de Óbitos por Grupo de Causa (SP - 2021)</b>',
    color='Total de Óbitos', color_continuous_scale='Blues'
)
fig_causas.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_causas.update_layout(template='plotly_white', showlegend=False, height=480)
fig_causas.show()

df_idade_sexo = df_analise[df_analise['SEXO_DESC'].isin(['Masculino', 'Feminino'])].dropna(subset=['IDADE_ANOS'])
fig_idade = px.histogram(
    df_idade_sexo, x='IDADE_ANOS', color='SEXO_DESC', barmode='overlay', nbins=35, opacity=0.6,
    title='<b>Distribuição Etária da Mortalidade por Sexo</b>',
    color_discrete_map={'Masculino': '#1f77b4', 'Feminino': '#e377c2'}
)
fig_idade.update_layout(template='plotly_white', height=450)
fig_idade.show()

df_mensal = df_analise.set_index('DTOBITO').resample('M').size().reset_index(name='Total_Obitos')
fig_tempo = px.line(df_mensal, x='DTOBITO', y='Total_Obitos', markers=True, title='<b>Evolução Mensal de Óbitos (SP - 2021)</b>')
fig_tempo.update_traces(line_color='#d62728', line_width=3)
fig_tempo.update_layout(template='plotly_white', height=400)
fig_tempo.show()

/tmp/ipykernel_2567/444867693.py:20: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



8. EXPORTAÇÃO PARA BI (Tableau / Power BI)

In [10]:
from google.colab import files

# 1. Base Granular Completa (1 linha por óbito)
colunas_dashboard = [
    'DTOBITO', 'COD_MUN_6', 'NOME_MUNICIPIO', 'MICRORREGIAO',
    'SEXO_DESC', 'RACA_DESC', 'IDADE_ANOS',
    'CAUSABAS', 'GRUPO_CID_REFINADO',
    'OBITO_PREMATURO_DCNT', 'OBITO_PREMATURO_EXTERNA'
]
df_tableau = df_analise[colunas_dashboard].copy()
df_tableau.to_parquet('mortalidade_sp_granular.parquet', index=False)
df_tableau.to_csv('mortalidade_sp_granular.csv', index=False, encoding='utf-8-sig')

# 2. Base Agregada Otimizada para Dashboard (~100k linhas)
df_analise['ANO_MES'] = df_analise['DTOBITO'].dt.to_period('M').astype(str)
df_tableau_agregado = df_analise.groupby([
    'ANO_MES', 'COD_MUN_6', 'NOME_MUNICIPIO', 'MICRORREGIAO',
    'SEXO_DESC', 'RACA_DESC', 'GRUPO_CID_REFINADO'
], as_index=False).agg(
    TOTAL_OBITOS=('DTOBITO', 'count'),
    OBITOS_PREMATUROS_DCNT=('OBITO_PREMATURO_DCNT', 'sum'),
    OBITOS_PREMATUROS_EXTERNA=('OBITO_PREMATURO_EXTERNA', 'sum'),
    IDADE_MEDIA=('IDADE_ANOS', 'mean')
)
df_tableau_agregado.to_csv('mortalidade_sp_agregado.csv', index=False, encoding='utf-8-sig')

print(f"✓ Exportações concluídas! Redução de {len(df_analise):,} para {len(df_tableau_agregado):,} linhas no agregado.")

# 3. Dispara o download automático do CSV agregado no seu navegador
files.download('mortalidade_sp_agregado.csv')

✓ Exportações concluídas! Redução de 431,616 para 102,388 linhas no agregado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>